In [2]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)


2025/10/14 02:20:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/10/14 02:20:38 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [6]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-camp/02-experiment-tracking/mlruns/1', creation_time=1760154421250, experiment_id='1', last_update_time=1760154421250, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1760152849355, experiment_id='0', last_update_time=1760152849355, lifecycle_stage='active', name='Default', tags={}>]

In [8]:
client.create_experiment(name="my-cool-experiment")

'2'

In [11]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids="1",
    filter_string="",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [14]:
runs

[<Run: data=<RunData: metrics={'best_iteration': 193.0,
  'rmse': 6.523024760146031,
  'stopped_iteration': 243.0,
  'validation-rmse': 6.522252969358939}, params={'custom_metric': 'None',
  'early_stopping_rounds': '50',
  'learning_rate': '0.4844888318402154',
  'max_depth': '77',
  'maximize': 'None',
  'min_child_weight': '9.198369208071277',
  'num_boost_round': '500',
  'objective': 'reg:linear',
  'reg_alpha': '0.019826555128805056',
  'reg_lambda': '0.004084671129828025',
  'seed': '42',
  'verbose_eval': 'True'}, tags={'mlflow.runName': 'adorable-auk-313',
  'mlflow.source.name': '/home/codespace/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',
  'mlflow.user': 'codespace'}>, info=<RunInfo: artifact_uri='/workspaces/mlops-camp/02-experiment-tracking/mlruns/1/727ac7e64ba449c7ae7a204770778f8b/artifacts', end_time=None, experiment_id='1', lifecycle_stage='active', run_id='727ac7e64ba449c7ae7a204770778f8b', run_name='adorable-auk-313',

In [13]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse:{run.data.metrics['rmse']:.4f}")

run id: 727ac7e64ba449c7ae7a204770778f8b, rmse:6.5230
run id: 59ddc5dc74ab4d029e288c02ec603b13, rmse:6.5230


KeyError: 'rmse'

In [16]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [18]:
run_id = "727ac7e64ba449c7ae7a204770778f8b"
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2025/10/14 02:47:19 WARNING mlflow.tracking._model_registry.fluent: Run with id 727ac7e64ba449c7ae7a204770778f8b has no artifacts at artifact path 'model', registering model based on models:/m-acdb7729a0584f408f21e2bdea5a71c9 instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1760410039408, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1760410039408, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='727ac7e64ba449c7ae7a204770778f8b', run_link=None, source='models:/m-acdb7729a0584f408f21e2bdea5a71c9', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [21]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1760410027582, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1760410039408, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1760410039408, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1760410039408, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='727ac7e64ba449c7ae7a204770778f8b', run_link=None, source='models:/m-acdb7729a0584f408f21e2bdea5a71c9', status='READY', status_message=None, tags={}, user_id=None, version=1>], name='nyc-taxi-regressor', tags={}>]

In [28]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version:{version.version}, stage:{version.current_stage}")

version:1, stage:None


/tmp/ipykernel_2335/270156577.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [30]:
client.transition_model_version_stage(
    name = model_name,
    version = 1,
    stage = "Staging",
    archive_existing_versions = False
)

/tmp/ipykernel_2335/1115275507.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1760410039408, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1760410714788, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='727ac7e64ba449c7ae7a204770778f8b', run_link=None, source='models:/m-acdb7729a0584f408f21e2bdea5a71c9', status='READY', status_message=None, tags={}, user_id=None, version=1>